# EDA — `bronze.trusts`

Trust metadata. Two jobs downstream: it supplies **manager** and **management group**,
which are what `dim_ticker` versions on as an SCD Type 2, and its `ticker` column defines
the universe the Yahoo pull requests.

What matters here is not price quality but **completeness** — a null manager becomes a
null in the dimension, so it needs to be a known quantity rather than a surprise in Gold.

## 1. Shape, blanks and duplicates

In [0]:
%sql
SELECT COUNT(*)                                             AS rows,
       COUNT(DISTINCT ticker)                               AS distinct_tickers,
       SUM(CASE WHEN ticker = '' THEN 1 ELSE 0 END)         AS blank_tickers,
       SUM(CASE WHEN manager = '' THEN 1 ELSE 0 END)        AS blank_managers,
       SUM(CASE WHEN management_group = '' THEN 1 ELSE 0 END) AS blank_groups
FROM `index-vs-trust-pipeline`.bronze.trusts;

In [0]:
%sql
-- A repeated ticker would break the join to prices and duplicate rows in the dimension.
SELECT ticker, COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.bronze.trusts
WHERE ticker <> ''
GROUP BY ticker
HAVING COUNT(*) > 1
ORDER BY rows DESC;

**120 rows, 119 distinct tickers, 2 blank.** The 119 counts the empty string as one
value: 118 real tickers plus the blank. The two blanks are Island Innovation and Witan.

If the duplicate query returns nothing, each real ticker appears once and the join to
prices is safe.

## 2. Does the metadata cover the trusts that actually have prices?

The universe that matters is the 100 symbols Yahoo returned, not the 118 requested. A
trust with prices but no manager is the case that hurts, because it reaches Gold.

In [0]:
%sql
WITH priced AS (
  SELECT DISTINCT source_ticker AS ticker
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
)
SELECT COUNT(*)                                                  AS priced_trusts,
       SUM(CASE WHEN t.manager <> '' THEN 1 ELSE 0 END)          AS with_manager,
       SUM(CASE WHEN t.manager = '' THEN 1 ELSE 0 END)           AS without_manager,
       SUM(CASE WHEN t.management_group <> '' THEN 1 ELSE 0 END) AS with_group,
       SUM(CASE WHEN t.ticker IS NULL THEN 1 ELSE 0 END)         AS priced_but_no_metadata
FROM priced p
LEFT JOIN `index-vs-trust-pipeline`.bronze.trusts t ON t.ticker = p.ticker;

In [0]:
%sql
-- Name the trusts that will carry a null manager into dim_ticker.
WITH priced AS (
  SELECT DISTINCT source_ticker AS ticker
  FROM `index-vs-trust-pipeline`.bronze.trust_prices_yf
)
SELECT t.ticker, t.trust_name, t.management_group
FROM priced p
JOIN `index-vs-trust-pipeline`.bronze.trusts t ON t.ticker = p.ticker
WHERE t.manager = ''
ORDER BY t.ticker;

Whatever this returns is the set of trusts that reach `dim_ticker` with a null manager.
They are **not dropped** — the universe stays honest, and the SCD2 simply has nothing to
version on for them until a manager appears. **Open for Silver** only in the sense of
confirming the null is carried rather than defaulted to a placeholder string.

## 3. What the SCD2 will actually version on

`dim_ticker` opens a new version when `manager`, `management_group` or `status` changes.
Worth knowing how many distinct values exist, and how shared they are.

In [0]:
%sql
SELECT management_group,
       COUNT(*) AS trusts
FROM `index-vs-trust-pipeline`.bronze.trusts
WHERE management_group <> ''
GROUP BY management_group
ORDER BY trusts DESC
LIMIT 10;

Management groups are genuinely shared — J.P. Morgan runs about 12 trusts and Baillie
Gifford about 10 — which is what earns `management_group` a place as a real grouping
attribute.

Individual managers are the opposite: roughly 227 people, of whom only 3 run more than
one trust. That is the evidence behind rejecting a `dim_manager` table — it would be a
column in disguise, and many-to-many on top.

## 4. The snapshot problem

This file has no start or end dates — it is a photograph of who manages what **today**.

In [0]:
%sql
DESCRIBE TABLE `index-vs-trust-pipeline`.bronze.trusts;

No `valid_from`, no `valid_to`, no `as_of_date`. So on the first run every trust becomes
version 1 and history accrues from the second run onwards.

That is normal behaviour for an SCD2 built against a live snapshot source, and it is
demonstrable rather than theoretical: change one manager in the CSV, re-run, and the
dimension shows two rows.

---

## Findings

| # | Finding | Status |
|---|---|---|
| 1.1 | 120 rows, 118 real tickers plus 2 blanks (Island Innovation, Witan). | **settled** |
| 1.2 | No duplicate tickers, so the join to prices cannot fan out. | **settled** |
| 1.3 | Of the 100 priced trusts, **97 have a manager and 3 do not**. All 100 have metadata. | **settled** |
| 1.4 | Trusts with prices but no manager keep a null rather than a placeholder. | **open** |
| 1.5 | Management groups are shared (J.P. Morgan ~12, Baillie Gifford ~10); individual managers are not (only 3 run more than one trust). Justifies `management_group` as an attribute and rejects `dim_manager`. | **settled** |
| 1.6 | The file is a snapshot with no validity dates, so SCD2 history starts at run 2. | **settled** |